# scIB-metrics across all integrations

* **Developed by:** Anna Maguza
* **Affilation:** CellZome, a GSK company
* **Created date:** 2026-05-08
* **Last modified date:** 2026-05-08

jax-based bio + batch metrics for every integrated embedding.


## Goal

Run scIB-metrics (jax variant) on every integrated embedding. Bio label = `cell_states`, batch label = `Study_name`. Drops cells with `cell_states == 'unknown'` (unlabelled LGR5 cells) so the label-aware metrics don't crash. Saves a combined CSV.

In [1]:
import os, sys, gc, json, datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

sc.settings.verbosity = 2
sc.settings.dpi = 120
sc.settings.dpi_save = 300
plt.rcParams.update({
    'savefig.bbox': 'tight', 'savefig.dpi': 300, 'figure.dpi': 120,
    'font.family': ['Arial', 'Helvetica', 'DejaVu Sans'], 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
})

/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [2]:
REPO         = Path('/Users/am336941/Library/CloudStorage/OneDrive-GSK/Desktop/Fetal_stem_cells_analysis')
DATA_OUT     = Path('/Users/am336941/PhD/data/Fetal_stem_cells_analysis_enhanced')
ATLAS_PATH   = Path('/Users/am336941/PhD/data/gut_data/gut_hs_all_datasets_full_annotated_AM_30102025_181544_raw.h5ad')
LGR5_DIR     = REPO / 'data' / 'LGR5_analysis'
ORTH_PATH    = Path('/Users/am336941/PhD/data/LGR5_analysis_data/human_mouse_orthologues_ensembl.txt')
DATA_OUT.mkdir(parents=True, exist_ok=True)

def stamp() -> str:
    return dt.datetime.now().strftime('%d%m%Y_%H%M%S')

import scib_metrics
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection


/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
adata_scanvi = sc.read_h5ad(DATA_OUT / 'integrated_obj_a_scanvi.h5ad')
adata_scpoli = sc.read_h5ad(DATA_OUT / 'integrated_obj_a_scpoli.h5ad')
adata_scvi = sc.read_h5ad(DATA_OUT / 'integrated_obj_a_scvi.h5ad')

In [11]:
adata_scvi

AnnData object with n_obs × n_vars = 102013 × 2000
    obs: 'organism', 'organism_part', 'cell_type', 'sample_id', 'Study_name', 'library_preparation_protocol', 'age_group', 'celltype', 'cell_states', 'gut_region', 'lgr5_status', 'species', 'source', 'n_genes_by_counts', 'total_counts', '_scvi_batch', '_scvi_labels'
    var: 'gene_id', 'gene_name', 'gene_type', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches'
    uns: 'Study_name_colors', '_scvi_manager_uuid', '_scvi_uuid', 'age_group_colors', 'cell_states_colors', 'gut_region_colors', 'hvg', 'lgr5_status_colors', 'neighbors', 'processing_history', 'umap'
    obsm: 'X_scvi', 'X_umap', '_scvi_extra_categorical_covs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [9]:
adata_scanvi

AnnData object with n_obs × n_vars = 102013 × 2000
    obs: 'organism', 'organism_part', 'cell_type', 'sample_id', 'Study_name', 'library_preparation_protocol', 'age_group', 'celltype', 'cell_states', 'gut_region', 'lgr5_status', 'species', 'source', 'n_genes_by_counts', 'total_counts', '_scvi_batch', '_scvi_labels'
    var: 'gene_id', 'gene_name', 'gene_type', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches'
    uns: 'Study_name_colors', '_scvi_manager_uuid', '_scvi_uuid', 'age_group_colors', 'cell_states_colors', 'gut_region_colors', 'hvg', 'lgr5_status_colors', 'neighbors', 'processing_history', 'umap'
    obsm: 'X_scanvi', 'X_scvi', 'X_umap', '_scvi_extra_categorical_covs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [7]:
adata_scpoli

AnnData object with n_obs × n_vars = 102013 × 2000
    obs: 'organism', 'organism_part', 'cell_type', 'sample_id', 'Study_name', 'library_preparation_protocol', 'age_group', 'celltype', 'cell_states', 'gut_region', 'lgr5_status', 'species', 'source', 'n_genes_by_counts', 'total_counts', '_scvi_batch', '_scvi_labels', 'conditions_combined'
    var: 'gene_id', 'gene_name', 'gene_type', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches'
    uns: 'Study_name_colors', '_scvi_manager_uuid', '_scvi_uuid', 'age_group_colors', 'cell_states_colors', 'gut_region_colors', 'hvg', 'lgr5_status_colors', 'neighbors', 'processing_history', 'umap'
    obsm: 'X_scanvi', 'X_scpoli', 'X_scvi', 'X_umap', '_scvi_extra_categorical_covs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [8]:
bio_metrics = BioConservation(isolated_labels=True, nmi_ari_cluster_labels_kmeans=True,
                              silhouette_label=True, clisi_knn=True)
batch_metrics = BatchCorrection(graph_connectivity=True, ilisi_knn=True,
                                kbet_per_label=False, pcr_comparison=True)

In [12]:
results = {}
obsm_keys = ['X_scpoli','X_scanvi','X_scvi','X_pca']
bm = Benchmarker(adata_scpoli, batch_key='Study_name', label_key='cell_states',embedding_obsm_keys=obsm_keys,
                         bio_conservation_metrics=bio_metrics,
                         batch_correction_metrics=batch_metrics,n_jobs=2)
bm.benchmark()
results= bm.get_results(min_max_scale=False, clean_names=True)

/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scanpy/preprocessing/_pca/__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(adata, mask_var, use_highly_variable)
/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scanpy/preprocessing/_pca/__init__.py:243: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  Version(ad.__version__) < Version("0.9")
Embeddings:   0%|          | 0/4 [00:00<?, ?it/s]/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scib_metrics/metrics/_graph_connectivity.py:32: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = p

In [14]:
results.to_csv("/Users/am336941/Library/CloudStorage/OneDrive-GSK/Desktop/Fetal_stem_cells_analysis/analysis_enhanced/3_integration/3a_human_only/figures/benchmark_results_all_human.csv")

In [15]:
plt.figure(figsize=(10, 6))
bm.plot_results_table(min_max_scale=False, show=False)
plt.savefig("/Users/am336941/Library/CloudStorage/OneDrive-GSK/Desktop/Fetal_stem_cells_analysis/analysis_enhanced/3_integration/3a_human_only/figures/scib_benchmark_all_human.png", dpi=300, bbox_inches='tight')
plt.close()

<Figure size 1200x720 with 0 Axes>